# LightGBM Fraud Detection Experiment
IEEE-CIS Fraud Detection Competition

## 0. Setup & MLflow Configuration

In [15]:

import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
import mlflow.lightgbm
import dagshub
import lightgbm as lgb

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score
from sklearn.feature_selection import VarianceThreshold

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

print('Libraries loaded successfully')

ImportError: No module named numpy

In [ ]:
from kaggle_secrets import UserSecretsClient

DAGSHUB_TOKEN    = UserSecretsClient().get_secret("DAGSHUB_TOKEN")
DAGSHUB_USERNAME = 'delibaaa'
REPO_NAME        = 'DELIBA-ML-ASSIGNMENT-2'

os.environ['MLFLOW_TRACKING_USERNAME'] = DAGSHUB_USERNAME
os.environ['MLFLOW_TRACKING_PASSWORD'] = DAGSHUB_TOKEN

MLFLOW_TRACKING_URI = f'https://dagshub.com/{DAGSHUB_USERNAME}/{REPO_NAME}.mlflow'
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

EXPERIMENT_NAME = 'LightGBM_Training'
mlflow.set_experiment(EXPERIMENT_NAME)

## 1. Data Loading

In [ ]:
DATA_PATH = '/kaggle/input/competitions/ieee-fraud-detection'

train_transaction = pd.read_csv(f'{DATA_PATH}/train_transaction.csv')
train_identity    = pd.read_csv(f'{DATA_PATH}/train_identity.csv')
test_transaction  = pd.read_csv(f'{DATA_PATH}/test_transaction.csv')
test_identity     = pd.read_csv(f'{DATA_PATH}/test_identity.csv')

train = train_transaction.merge(train_identity, on='TransactionID', how='left')
test  = test_transaction.merge(test_identity, on='TransactionID', how='left')

print(f'Train shape: {train.shape}')
print(f'Test shape:  {test.shape}')
print(f'Fraud rate:  {train["isFraud"].mean():.4f}')

## 2. Cleaning

In [ ]:
with mlflow.start_run(run_name='LightGBM_Cleaning'):

    nan_pct = train.isnull().mean().sort_values(ascending=False)
    high_nan_cols = nan_pct[nan_pct > 0.9].index.tolist()
    print(f'Columns with >90% NaN: {len(high_nan_cols)}')

    train_clean = train.drop(columns=high_nan_cols)
    test_clean  = test.drop(columns=high_nan_cols, errors='ignore')

    dup_count = train_clean.duplicated().sum()
    train_clean = train_clean.drop_duplicates()
    print(f'Removed duplicates: {dup_count}')

    constant_cols = [c for c in train_clean.columns
                     if train_clean[c].nunique(dropna=False) <= 1]
    train_clean = train_clean.drop(columns=constant_cols)
    test_clean  = test_clean.drop(columns=constant_cols, errors='ignore')
    print(f'Removed constant cols: {len(constant_cols)}')

    mlflow.log_param('dropped_high_nan_cols', len(high_nan_cols))
    mlflow.log_param('dropped_constant_cols', len(constant_cols))
    mlflow.log_param('removed_duplicates', dup_count)
    mlflow.log_metric('train_rows_after_cleaning', len(train_clean))
    mlflow.log_metric('train_cols_after_cleaning', train_clean.shape[1])

    print(f'Shape after cleaning: {train_clean.shape}')

## 3. Feature Engineering

In [ ]:
with mlflow.start_run(run_name='LightGBM_Feature_Engineering'):

    def feature_engineering(df):
        df = df.copy()

        START_DATE = pd.Timestamp('2017-11-30')
        df['TransactionDay']    = (df['TransactionDT'] // (3600 * 24)).astype(int)
        df['TransactionHour']   = (df['TransactionDT'] // 3600 % 24).astype(int)
        df['TransactionWeekDay']= ((df['TransactionDT'] // (3600 * 24)) % 7).astype(int)

        df['TransactionAmt_log'] = np.log1p(df['TransactionAmt'])
        df['TransactionAmt_decimal'] = df['TransactionAmt'] - df['TransactionAmt'].astype(int)

        for col in ['card1', 'card2', 'addr1', 'P_emaildomain', 'R_emaildomain']:
            if col in df.columns:
                freq = df[col].value_counts(normalize=True)
                df[f'{col}_freq'] = df[col].map(freq).fillna(0)

        if 'card1' in df.columns:
            card1_amt_mean = df.groupby('card1')['TransactionAmt'].transform('mean')
            card1_amt_std  = df.groupby('card1')['TransactionAmt'].transform('std').fillna(0)
            df['card1_amt_mean'] = card1_amt_mean
            df['card1_amt_std']  = card1_amt_std
            df['card1_amt_diff'] = df['TransactionAmt'] - card1_amt_mean

        if 'P_emaildomain' in df.columns:
            df['P_email_suffix'] = df['P_emaildomain'].str.split('.').str[-1].fillna('unknown')

        return df

    train_fe = feature_engineering(train_clean)
    test_fe  = feature_engineering(test_clean)

    new_features = [
        'TransactionDay', 'TransactionHour', 'TransactionWeekDay',
        'TransactionAmt_log', 'TransactionAmt_decimal',
        'card1_freq', 'card1_amt_mean', 'card1_amt_std', 'card1_amt_diff',
        'P_email_suffix'
    ]

    mlflow.log_param('new_features_added', len([f for f in new_features if f in train_fe.columns]))
    mlflow.log_param('feature_engineering_methods',
                     'time_features,log_transform,frequency_encoding,aggregations,email_parsing')
    mlflow.log_metric('total_features_after_fe', train_fe.shape[1])

    print(f'Shape after FE: {train_fe.shape}')

## 4. Feature Selection

In [ ]:
with mlflow.start_run(run_name='LightGBM_Feature_Selection'):

    TARGET = 'isFraud'
    DROP_COLS = ['TransactionID', 'TransactionDT', TARGET]

    feature_cols = [c for c in train_fe.columns if c not in DROP_COLS]

    cat_cols = train_fe[feature_cols].select_dtypes(include='object').columns.tolist()
    num_cols = train_fe[feature_cols].select_dtypes(exclude='object').columns.tolist()

    corr_matrix = train_fe[num_cols].corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    high_corr_cols = [col for col in upper.columns if any(upper[col] > 0.95)]
    print(f'High correlation cols dropped: {len(high_corr_cols)}')

    num_cols_filtered = [c for c in num_cols if c not in high_corr_cols]

    X_sample = train_fe[num_cols_filtered + cat_cols].copy()
    y_sample = train_fe[TARGET]

    for col in cat_cols:
        X_sample[col] = X_sample[col].astype('category').cat.codes

    X_sample = X_sample.fillna(-999)

    quick_model = lgb.LGBMClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    quick_model.fit(X_sample, y_sample)

    importance_df = pd.DataFrame({
        'feature': num_cols_filtered + cat_cols,
        'importance': quick_model.feature_importances_
    }).sort_values('importance', ascending=False)

    selected_features = importance_df[importance_df['importance'] > 10]['feature'].tolist()
    print(f'Selected features: {len(selected_features)} / {len(feature_cols)}')

    mlflow.log_param('initial_features', len(feature_cols))
    mlflow.log_param('high_corr_dropped', len(high_corr_cols))
    mlflow.log_param('selected_features_count', len(selected_features))
    mlflow.log_param('selection_method', 'correlation_threshold_0.95 + lgbm_importance_threshold_10')

    importance_df.to_csv('/tmp/lgbm_feature_importance.csv', index=False)
    mlflow.log_artifact('/tmp/lgbm_feature_importance.csv')

    print(f'Top 10 features:\n{importance_df.head(10).to_string()}')

## 5. Cross Validation (Baseline)

In [ ]:
with mlflow.start_run(run_name='LightGBM_CV_Baseline'):

    X = train_fe[selected_features].copy()
    y = train_fe[TARGET]

    cat_in_selected = [c for c in selected_features if train_fe[c].dtype == 'object']
    for col in cat_in_selected:
        X[col] = X[col].astype('category').cat.codes
    X = X.fillna(-999)

    baseline_params = {
        'n_estimators': 500,
        'learning_rate': 0.05,
        'num_leaves': 64,
        'max_depth': -1,
        'min_child_samples': 20,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'random_state': 42,
        'n_jobs': -1,
        'class_weight': 'balanced'
    }

    model = lgb.LGBMClassifier(**baseline_params)

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = cross_val_score(model, X, y, cv=skf, scoring='roc_auc', n_jobs=-1)

    print(f'CV AUC scores: {cv_scores}')
    print(f'Mean AUC: {cv_scores.mean():.5f} (+/- {cv_scores.std():.5f})')

    mlflow.log_params(baseline_params)
    mlflow.log_metric('cv_auc_mean', cv_scores.mean())
    mlflow.log_metric('cv_auc_std', cv_scores.std())
    for i, score in enumerate(cv_scores):
        mlflow.log_metric(f'cv_fold_{i+1}_auc', score)

## 6. Hyperparameter Tuning (Optuna)

In [ ]:
with mlflow.start_run(run_name='LightGBM_Tuning'):

    def objective(trial):
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 200, 1000),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
            'num_leaves': trial.suggest_int('num_leaves', 31, 256),
            'max_depth': trial.suggest_int('max_depth', 3, 12),
            'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
            'subsample': trial.suggest_float('subsample', 0.5, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
            'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
            'random_state': 42,
            'n_jobs': -1
        }
        model = lgb.LGBMClassifier(**params)
        skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
        scores = cross_val_score(model, X, y, cv=skf, scoring='roc_auc')
        return scores.mean()

    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=30, show_progress_bar=True)

    best_params = study.best_params
    best_params['random_state'] = 42
    best_params['n_jobs'] = -1

    print(f'Best AUC: {study.best_value:.5f}')
    print(f'Best params: {best_params}')

    mlflow.log_params(best_params)
    mlflow.log_metric('best_cv_auc', study.best_value)
    mlflow.log_param('n_trials', 30)

## 7. Overfit / Underfit Analysis

In [ ]:
with mlflow.start_run(run_name='LightGBM_Overfit_Experiment'):

    overfit_model = lgb.LGBMClassifier(
        n_estimators=2000,
        num_leaves=512,
        max_depth=-1,
        min_child_samples=1,
        learning_rate=0.3,
        random_state=42
    )

    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    train_aucs, val_aucs = [], []

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
        overfit_model.fit(X_tr, y_tr)
        train_aucs.append(roc_auc_score(y_tr, overfit_model.predict_proba(X_tr)[:,1]))
        val_aucs.append(roc_auc_score(y_val, overfit_model.predict_proba(X_val)[:,1]))

    print('[OVERFIT] Train AUC: %.5f | Val AUC: %.5f' % (np.mean(train_aucs), np.mean(val_aucs)))
    print('Gap: %.5f' % (np.mean(train_aucs) - np.mean(val_aucs)))

    mlflow.log_metric('train_auc', np.mean(train_aucs))
    mlflow.log_metric('val_auc', np.mean(val_aucs))
    mlflow.log_metric('overfit_gap', np.mean(train_aucs) - np.mean(val_aucs))
    mlflow.log_param('model_type', 'overfit_intentional')


In [ ]:
with mlflow.start_run(run_name='LightGBM_Underfit_Experiment'):

    underfit_model = lgb.LGBMClassifier(
        n_estimators=10,
        num_leaves=4,
        max_depth=2,
        min_child_samples=200,
        learning_rate=0.01,
        random_state=42
    )

    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    cv_scores_underfit = cross_val_score(underfit_model, X, y, cv=skf, scoring='roc_auc')

    print('[UNDERFIT] CV AUC: %.5f' % cv_scores_underfit.mean())

    mlflow.log_metric('cv_auc_mean', cv_scores_underfit.mean())
    mlflow.log_param('model_type', 'underfit_intentional')

NameError: name 'mlflow' is not defined

## 8. Final Pipeline Training & MLflow Registration

In [ ]:
with mlflow.start_run(run_name='LightGBM_Final') as run:

    from sklearn.pipeline import Pipeline
    from sklearn.compose import ColumnTransformer
    from sklearn.preprocessing import OrdinalEncoder
    from sklearn.impute import SimpleImputer

    X_raw = train_fe[selected_features].copy()
    cat_feats = X_raw.select_dtypes(include='object').columns.tolist()
    num_feats = X_raw.select_dtypes(exclude='object').columns.tolist()

    numeric_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='constant', fill_value=-999))
    ])

    categorical_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
        ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
    ])

    preprocessor = ColumnTransformer(transformers=[
        ('num', numeric_transformer, num_feats),
        ('cat', categorical_transformer, cat_feats)
    ])

    best_lgbm = lgb.LGBMClassifier(**best_params)

    final_pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', best_lgbm)
    ])

    X_raw_filled = train_fe[selected_features].copy()
    y_final = train_fe[TARGET]

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    final_cv_scores = cross_val_score(final_pipeline, X_raw_filled, y_final,
                                       cv=skf, scoring='roc_auc', n_jobs=-1)

    print(f'Final Pipeline CV AUC: {final_cv_scores.mean():.5f} (+/- {final_cv_scores.std():.5f})')

    final_pipeline.fit(X_raw_filled, y_final)

    mlflow.log_params(best_params)
    mlflow.log_metric('final_cv_auc_mean', final_cv_scores.mean())
    mlflow.log_metric('final_cv_auc_std', final_cv_scores.std())
    mlflow.log_param('selected_features', str(selected_features))
    mlflow.log_param('n_selected_features', len(selected_features))

    mlflow.sklearn.log_model(
        sk_model=final_pipeline,
        artifact_path='lgbm_pipeline',
        registered_model_name='LightGBM_Fraud_Pipeline'
    )

    print(f'Model registered! Run ID: {run.info.run_id}')
    print(f'Model URI: runs:/{run.info.run_id}/lgbm_pipeline')

SyntaxError: invalid syntax (<ipython-input-4-5474467e2ea4>, line 40)

In [ ]:
import json
with open('lgbm_selected_features.json', 'w') as f:
    json.dump(selected_features, f)
print('Selected features saved to lgbm_selected_features.json')